#Naïve Bayes in Python

This notebook uses the naive Bayes classifier to classify movie reviews in the NLTK Movie Reviews data set. This involves identifying the sentiment (positive or negative) expressed in text.


In [1]:
#Import necessary libraries
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import nltk
from nltk.corpus import movie_reviews
import random
import pandas as pd

In [2]:
#Download required NLTK data (run once)
nltk.download('movie_reviews',quiet = True)

True

In [3]:
#Load the movie reviews data
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

print(f"Number of documents: {len(documents)}")
print(f"Example document words: {documents[0][0][:20]}")
print(f"Example document category: {documents[0][1]}")

Number of documents: 2000
Example document words: ['plot', ':', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', ',', 'drink', 'and', 'then', 'drive', '.', 'they', 'get', 'into', 'an']
Example document category: neg


In [4]:
#Load documents as DataFrame
df = pd.DataFrame(documents)
print(df.head())

                                                   0    1
0  [plot, :, two, teen, couples, go, to, a, churc...  neg
1  [the, happy, bastard, ', s, quick, movie, revi...  neg
2  [it, is, movies, like, these, that, make, a, j...  neg
3  [", quest, for, camelot, ", is, warner, bros, ...  neg
4  [synopsis, :, a, mentally, unstable, man, unde...  neg



Randomly shuffle the documents by using `.random.shuffle()`.


In [ ]:
#Shuffle the documents for randomness
random.shuffle(documents)

####Finding word frequencies
To build a machine learning model for text classification, such as identifying positive or negative movie reviews, relevant features must be extracted from raw text.

Using NLTK, the data set is processed by computing a frequency distribution of all words and normalised to lowercase for consistency. The 2,000 most frequent words are then selected as the model’s feature set.

In [5]:
all_words = nltk.FreqDist(w.lower() for w in movie_reviews.words()) # used to find word frequencies (w.lower used to normalise words to lowercase)
word_features = list(all_words)[:2000] #assign frequency distribution to words
print("The most common words are ", word_features[:10])

The most common words are  [',', 'the', '.', 'a', 'and', 'of', 'to', "'", 'is', 'in']


The `document_features` function converts a text document into a structured feature vector by checking for the presence of a the most frequent terms in the data set (found above).

The function returns a dict where each key corresponds to a word and outputs a Boolean indicating whether the word appears in the document.

In [6]:
def document_features(document):
    document_words = set(w.lower() for w in document)
    features = {}
    for word in word_features:
        features[f'contains({word})'] = (word in document_words)
    return features

The following code represents each document as a fixed set of features paired with its sentiment label.
- This is stored as feature-label pairs in `featuresets`

In [7]:
#using list comprehesion
featuresets = [(document_features(doc[0]), doc[1]) for doc in documents]

#this is the alternative to list comprehension (using for loop)
if False: #used to stop code running
  for doc in range(len(documents)):
    features = document_features(doc)
    featuresets.append(features)

print(featuresets[0])

({'contains(,)': True, 'contains(the)': True, 'contains(.)': True, 'contains(a)': True, 'contains(and)': True, 'contains(of)': True, 'contains(to)': True, "contains(')": True, 'contains(is)': True, 'contains(in)': True, 'contains(s)': True, 'contains(")': True, 'contains(it)': True, 'contains(that)': True, 'contains(-)': True, 'contains())': True, 'contains(()': True, 'contains(as)': True, 'contains(with)': True, 'contains(for)': True, 'contains(his)': True, 'contains(this)': True, 'contains(film)': True, 'contains(i)': True, 'contains(he)': True, 'contains(but)': True, 'contains(on)': True, 'contains(are)': True, 'contains(t)': True, 'contains(by)': True, 'contains(be)': True, 'contains(one)': True, 'contains(movie)': True, 'contains(an)': True, 'contains(who)': True, 'contains(not)': True, 'contains(you)': True, 'contains(from)': True, 'contains(at)': False, 'contains(was)': False, 'contains(have)': True, 'contains(they)': True, 'contains(has)': True, 'contains(her)': True, 'contains

####Splitting the dataset into training and test sets
The data set of 2,000 documents is split into:
- 1, 800 documents for training (featuresets[:1800]), which is 90 per cent of the data
- 200 documents for testing (featuresets[1800:]), which is the remaining 10 per cent of the data


In [10]:
train_set = featuresets[:1800]
test_set = featuresets[1800:]
print(len(train_set), len(test_set))

1800 200


NLTK classifiers, such as `NaiveBayesClassifier`, require data in the form of (features, label) tuples. Unlike libraries such as scikit-learn, which separate features (X) and labels (y), NLTK’s format is already aligned with this structure. This makes splitting into `train_set` and `test_set` straightforward and directly compatible with NLTK’s training and evaluation methods.

To train the classifier, use `NaiveBayesClassifier` from NLTK: https://www.nltk.org/_modules/nltk/classify/naivebayes.html.

In [11]:
classifier = nltk.NaiveBayesClassifier.train(train_set)
print(classifier.show_most_informative_features(10))

# pos : neg ratio is the likelihood ratio between the 2 sentiment classes (positive = pos, negative = neg)
## eg. contains(outstanding) > if the review contains 'outstanding' it is 13.3x more likely to be positive than negative

Most Informative Features
         contains(mulan) = True              pos : neg    =     11.2 : 1.0
   contains(outstanding) = True              pos : neg    =     11.0 : 1.0
        contains(seagal) = True              neg : pos    =     10.9 : 1.0
         contains(damon) = True              pos : neg    =      7.1 : 1.0
         contains(flynt) = True              pos : neg    =      7.1 : 1.0
   contains(wonderfully) = True              pos : neg    =      7.0 : 1.0
          contains(lame) = True              neg : pos    =      6.4 : 1.0
         contains(awful) = True              neg : pos    =      5.3 : 1.0
        contains(wasted) = True              neg : pos    =      5.3 : 1.0
        contains(poorly) = True              neg : pos    =      5.1 : 1.0
None


####Test the classifier on the test set

In [ ]:
accuracy = nltk.classify.accuracy(classifier, test_set)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.75


An accuracy of 75% acts as a benchmark.


##Sentiment classifier integrating several models

The following classifier integrates scores from multiple models:
- Naive Bayes
- Logistic regression
- Support vector machine (SVMs)

A `VotingClassifier` ensemble combines their predictions, improving both accuracy and stability.
- `VotingClassifier` predicts output based on the aggregate majority of votes from each model for each output class
   - 'hard' = predicts class as the class with the highest majority of votes across al models
   - 'soft' = average probabilities of the classes determine the final predicted class

In [13]:
import nltk
from nltk.corpus import movie_reviews
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

#Separate the text and the labels
texts = [" ".join(doc) for doc, label in documents] # Join words into a string
labels = [label for doc, label in documents]

#Split the data into train and test sets (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.1, random_state=42)

#CountVectoriser converts words into a vector based on their frequency ("tokenisation")
vectorizer = CountVectorizer(stop_words='english', max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

#Define the classifiers to try
classifiers = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC()
}

#Train and evaluate each classifier
for name, clf in classifiers.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.4f}")

Multinomial Naive Bayes Accuracy: 0.7750
Logistic Regression Accuracy: 0.8100
Linear SVM Accuracy: 0.8050


In [14]:
from sklearn.ensemble import VotingClassifier
#Create a VotingClassifier ensemble
ensemble = VotingClassifier(
    estimators=[
        ('nb', classifiers["Multinomial Naive Bayes"]),
        ('lr', classifiers["Logistic Regression"]),
        ('svm', classifiers["Linear SVM"])
        ],
    voting='hard'  # 'soft' for soft voting
)

#Train the ensemble
ensemble.fit(X_train_vec, y_train)

#Predict on the test set
y_pred = ensemble.predict(X_test_vec)

#Evaluate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Ensemble model accuracy: {accuracy:.4f}')

Ensemble model accuracy: 0.8150
